In [ ]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import euclidean_distances
import os

# -------------------------- 1. 0类：精准随机过采样（源头控制1000个） --------------------------
def exact_oversample_minority(X: np.ndarray, y: np.ndarray, target_num: int, cls: int) -> tuple:
    """
    0类精准过采样：计算需新增样本数，随机复制原始样本，确保最终数量=target_num
    X: 0类特征矩阵
    y: 0类标签矩阵（全为cls）
    target_num: 目标数量（1000）
    cls: 类别标签（0）
    返回：数量达标的(X_balanced, y_balanced)
    """
    current_count = len(X)
    # 计算需新增的样本数量
    n_needed = target_num - current_count
    if n_needed <= 0:
        # 数量过多：随机截取到target_num个（无放回）
        keep_idx = np.random.choice(current_count, size=target_num, replace=False)
        return X[keep_idx], y[keep_idx]
    
    # 数量不足：随机复制n_needed个样本（有放回，确保分布均匀）
    # 按原始样本索引权重抽样（避免重复复制同一批样本）
    copy_idx = np.random.choice(current_count, size=n_needed, replace=True, p=np.ones(current_count)/current_count)
    X_copy = X[copy_idx]
    y_copy = np.full(n_needed, cls)
    
    # 合并原始样本和复制样本（总量=target_num）
    X_balanced = np.vstack([X, X_copy])
    y_balanced = np.hstack([y, y_copy])
    
    # 验证数量（源头确保精准）
    assert len(X_balanced) == target_num, f"0类过采样后数量={len(X_balanced)}，目标={target_num}"
    return X_balanced, y_balanced
 
# -------------------------- 2. 1类：精准分层采样（源头控制1000个） --------------------------
def exact_stratified_sampling(X: np.ndarray, y: np.ndarray, target_num: int, n_clusters: int = 15) -> tuple:
    """
    1类精准分层采样：按簇内样本占比分配名额，确保总量=target_num（无补全）
    X: 1类特征矩阵
    y: 1类标签矩阵（全为1）
    target_num: 目标数量（1000）
    n_clusters: 聚类数量（15个，保证簇内分布均匀）
    返回：数量达标的(X_sampled, y_sampled)
    """
    # 步骤1：1类样本聚类（保留分布特征）
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)
    
    # 步骤2：统计每个簇的样本数量及占比
    cluster_counts = {c: len(X[cluster_labels == c]) for c in range(n_clusters)}
    total_samples = len(X)
    
    # 步骤3：按占比分配采样名额（确保总和=target_num）
    sample_quota = {}
    remaining = target_num  # 剩余待分配名额
    
    # 第一轮：按比例分配基础名额（每个簇至少1个，避免小簇被忽略）
    for c in cluster_counts:
        ratio = cluster_counts[c] / total_samples
        base_quota = int(ratio * target_num)
        base_quota = max(base_quota, 1)  # 确保每个簇有采样名额
        sample_quota[c] = base_quota
        remaining -= base_quota
    
    # 第二轮：分配剩余名额（按簇内样本数降序，保证分布均衡）
    clusters_sorted = sorted(cluster_counts.keys(), key=lambda x: cluster_counts[x], reverse=True)
    for c in clusters_sorted:
        if remaining <= 0:
            break
        sample_quota[c] += 1
        remaining -= 1
    
    # 步骤4：按名额无放回采样
    X_sampled = []
    for c in sample_quota:
        cluster_samples = X[cluster_labels == c]
        quota = sample_quota[c]
        # 确保名额不超过簇内样本数（极端情况：簇内样本不足）
        quota = min(quota, len(cluster_samples))
        sampled_idx = np.random.choice(len(cluster_samples), size=quota, replace=False)
        X_sampled.extend(cluster_samples[sampled_idx])
    
    # 转换为数组并生成标签
    X_sampled = np.array(X_sampled)
    y_sampled = np.full(len(X_sampled), 1)
    
    # 验证数量（源头确保精准）
    assert len(X_sampled) == target_num, f"1类分层采样后数量={len(X_sampled)}，目标={target_num}"
    return X_sampled, y_sampled

# -------------------------- 3. 2类：精准CSMOTE（源头控制1000个） --------------------------
def exact_csmote(X_min: np.ndarray, X_maj: np.ndarray, target_num: int = 1000, k: int = 6, max_round: int = 3) -> tuple:
    """
    2类精准CSMOTE：优化参数+多轮合成，确保总量=target_num（无补全）
    X_min: 2类特征矩阵（少数类）
    X_maj: 1类特征矩阵（多数类，用于筛选有效样本）
    target_num: 目标数量（1000）
    k: 聚类数量（6个，覆盖更多子分布）
    max_round: 多轮合成轮次（3轮，逐步逼近目标）
    返回：数量达标的(X_min_balanced, y_min_balanced)
    """
    current_count = len(X_min)
    n_needed_total = target_num - current_count
    if n_needed_total <= 0:
        # 数量过多：截取到target_num个
        keep_idx = np.random.choice(current_count, size=target_num, replace=False)
        return X_min[keep_idx], np.full(target_num, 2)
    
    # 步骤1：少数类聚类+初始化合成样本（簇心先加入）
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_min)
    centroids = kmeans.cluster_centers_
    synthetic_samples = list(centroids)  # 簇心作为初始合成样本
    
    # 步骤2：多轮合成（每轮调整比例，提升合成效率）
    remaining = n_needed_total
    for _ in range(max_round):
        if remaining <= 0:
            break
        
        # 按簇合成（优先合成有效样本多的簇）
        for cluster_id in range(k):
            cluster_X = X_min[cluster_labels == cluster_id]
            if len(cluster_X) < 2:  # 簇内样本过少，跳过
                continue
            
            # 筛选有效样本（放宽条件，增加有效样本量）
            centroid = centroids[cluster_id]
            dist_min_cent = euclidean_distances(cluster_X, centroid.reshape(1, -1)).flatten()
            dist_maj_cent = euclidean_distances(X_maj, centroid.reshape(1, -1)).flatten()
            min_dist_maj = np.min(dist_maj_cent)
            valid_mask = dist_min_cent < (min_dist_maj * 1.05)  # 轻微放宽，增加有效样本
            valid_X = cluster_X[valid_mask]
            if len(valid_X) < 2:
                continue
            
            # 按剩余需求分配本轮合成名额
            cluster_ratio = len(valid_X) / len(X_min)
            n_synth = int(remaining * cluster_ratio)
            n_synth = max(n_synth, 2)  # 每轮至少合成2个，避免效率低
            n_synth = min(n_synth, remaining)  # 不超过剩余需求
            
            # 合成新样本（增加多样性，避免单一插值）
            for _ in range(n_synth):
                # 随机选2个有效样本，在簇心与样本间插值
                idx1, idx2 = np.random.choice(len(valid_X), size=2, replace=False)
                sample = valid_X[idx1] if np.random.random() < 0.5 else valid_X[idx2]
                rand = np.random.uniform(0.2, 0.8)  # 避开0/1，增加样本多样性
                new_sample = centroid + rand * (sample - centroid)
                synthetic_samples.append(new_sample)
            
            remaining -= n_synth
            if remaining <= 0:
                break
    
    # 步骤3：合并原始样本和合成样本（确保总量=target_num）
    X_min_synth = np.vstack([X_min, np.array(synthetic_samples)])
    # 若合成过多，截取前target_num个；极端不足则用合成样本补（避免复制原始样本）
    if len(X_min_synth) >= target_num:
        X_min_balanced = X_min_synth[:target_num]
    else:
        need = target_num - len(X_min_synth)
        copy_idx = np.random.choice(len(X_min_synth), size=need, replace=False)
        X_min_balanced = np.vstack([X_min_synth, X_min_synth[copy_idx]])
    
    y_min_balanced = np.full(len(X_min_balanced), 2)
    
    # 验证数量（源头确保精准）
    assert len(X_min_balanced) == target_num, f"2类CSMOTE后数量={len(X_min_balanced)}，目标={target_num}"
    return X_min_balanced, y_min_balanced

# -------------------------- 4. 主流程：三类精准平衡+结果保存 --------------------------
if __name__ == "__main__":
    # 路径配置
    train_path = r"H:\图像标记\合并_513_627.xlsx"  # 原始训练集路径
    save_dir = r"H:\图像标记\上下采样"             # 结果保存路径
    save_filename = "三类精准平衡后_训练集.xlsx"    # 保存文件名
    save_path = os.path.join(save_dir, save_filename)
    target_num = 1000  # 三类统一目标数量
    
    # 1. 创建保存目录（若不存在）
    os.makedirs(save_dir, exist_ok=True)
    
    # 2. 读取并验证原始数据
    df = pd.read_excel(train_path)
    label_col = "Label"  # 替换为你的标签列名（如"类别"）
    if label_col not in df.columns:
        raise ValueError(f"标签列 '{label_col}' 不存在！实际列名：{df.columns.tolist()}")
    
    # 分离特征和标签
    X = df.drop(columns=[label_col]).values
    y = df[label_col].values
    print("原始类别分布：", {cls: np.sum(y == cls) for cls in [0, 1, 2]})
    
    # 3. 按类别执行精准平衡（无后续补全）
    ## 3.1 0类：精准随机过采样
    mask_0 = (y == 0)
    X_0, y_0 = X[mask_0], y[mask_0]
    X_0_balanced, y_0_balanced = exact_oversample_minority(X_0, y_0, target_num, cls=0)
    print(f"0类平衡后：{len(X_0_balanced)}个（原始{len(X_0)}个 + 新增{target_num-len(X_0)}个）")
    
    ## 3.2 1类：精准分层采样
    mask_1 = (y == 1)
    X_1, y_1 = X[mask_1], y[mask_1]
    X_1_balanced, y_1_balanced = exact_stratified_sampling(X_1, y_1, target_num, n_clusters=15)
    print(f"1类平衡后：{len(X_1_balanced)}个（分层采样{target_num}个）")
    
    ## 3.3 2类：精准CSMOTE
    mask_2 = (y == 2)
    X_2, y_2 = X[mask_2], y[mask_2]
    X_2_balanced, y_2_balanced = exact_csmote(X_2, X_1, target_num, k=6, max_round=3)
    print(f"2类平衡后：{len(X_2_balanced)}个（原始{len(X_2)}个 + 合成{target_num-len(X_2)}个）")
    
    # 4. 合并三类样本并验证
    X_balanced = np.vstack([X_0_balanced, X_1_balanced, X_2_balanced])
    y_balanced = np.hstack([y_0_balanced, y_1_balanced, y_2_balanced])
    
    # 最终验证（必须全为1000个）
    final_counts = {cls: np.sum(y_balanced == cls) for cls in [0, 1, 2]}
    print("\n最终三类平衡分布：", final_counts)
    assert all(count == target_num for count in final_counts.values()), \
        f"三类未全部达到1000个！当前分布：{final_counts}"
    
    # 5. 保存平衡训练集（保留原始列名）
    balanced_df = pd.DataFrame(X_balanced, columns=df.drop(columns=[label_col]).columns)
    balanced_df.insert(0, label_col, y_balanced)
    balanced_df.to_excel(save_path, index=False, engine="openpyxl")
    print(f"\n平衡训练集已保存至：{save_path}")